In [1]:
import jax.numpy as jnp
import numpy as np
import hj_reachability as hj
import matplotlib.pyplot as plt

# Air3d Dynamics:

$x_1$: position of the pursuer relative to evader

$x_2$: position of the evader relative to the pursuer

$x_3$: relative heading angle (pursuer heading - evader_heading)

$v_e$: speed of evader

$v_p$: speed of pursuer

$w_e$: evader turn rate (control $u$)

$w_p$: pursuer turn rate (disturbance $d$)

$$\dot{x} = f(x) + G(x)u + W(x)d$$

$$\dot{x} =
\begin{bmatrix}
\dot{x_1} \\
\dot{x_2}\\
\dot{x_3}
\end{bmatrix}
=
\begin{bmatrix}
-v_e + v_p\cos(x_3) + w_ex_2 \\
v_p\sin(x_3) - w_ex_1\\
w_p - w_e
\end{bmatrix}
$$

Control Jacobian:
$$\begin{bmatrix}
x_2\\
-x_1\\
-1
\end{bmatrix}$$

Disturbance Jacobian:
$$\begin{bmatrix}
0\\
0\\
1
\end{bmatrix}$$

HJI:
$$\frac{\partial V}{\partial t} + \min_d\max_u\nabla V^\top(f(x)+G(x)u+W(x)d) = 0$$
$$V(0, x) = g(x)$$

# Define Air3d Dynamics

In [2]:
class Air3d(hj.ControlAndDisturbanceAffineDynamics):
    def __init__(self, evader_speed=5.0, pursuer_speed=5.0, evader_max_turn_rate=1.0, pursuer_max_turn_rate=1.0, control_mode="max", disturbance_mode="min"):
        self.evader_speed = evader_speed
        self.pursuer_speed = pursuer_speed

        control_space = hj.sets.Box(jnp.array([-evader_max_turn_rate]), jnp.array([evader_max_turn_rate]))
        disturbance_space = hj.sets.Box(jnp.array([-pursuer_max_turn_rate]), jnp.array([pursuer_max_turn_rate]))

        super().__init__(control_mode, disturbance_mode, control_space, disturbance_space)
    
    def open_loop_dynamics(self, state, time):
        _, _, psi = state 
        v_a, v_b = self.evader_speed, self.pursuer_speed
        return jnp.array([-v_a + v_b * jnp.cos(psi), v_b * jnp.sin(psi), 0.0])

    def control_jacobian(self, state, time):
        x, y, _ = state
        return jnp.array([[y], [-x], [-1.0]])

    def disturbance_jacobian(self, state, time):
        return jnp.array([[0.0], [0.0], [1.0]])

# Define time and grid discretization

In [3]:
time = -1.0 # time of simulation
time_discretization = 20

x1_discretization = 201
x2_discretization = 201
x3_discretization = 201

# Change controller or disturbances

In [4]:
evader_speed = 0.75
pursuer_speed = 0.75
evader_max_turn_rate = 3.0
pursuer_max_turn_rate = 3.0

# Instantiate grid and dynamics

In [5]:
x1_min, x1_max = -1, 1.0
x2_min, x2_max = -1.0, 1.0
x3_min, x3_max = 0.0, 2*np.pi

times = np.linspace(0.0, time, time_discretization)
dynamics = Air3d(evader_speed=evader_speed, pursuer_speed=pursuer_speed, evader_max_turn_rate=evader_max_turn_rate, pursuer_max_turn_rate=pursuer_max_turn_rate)
grid = hj.Grid.from_lattice_parameters_and_boundary_conditions(hj.sets.Box(np.array([x1_min, x2_min, x3_min]),np.array([x1_max, x2_max, x3_max]),), shape=(x1_discretization, x2_discretization, x3_discretization), periodic_dims=(2,))

# Create the target set

In [6]:
positions = grid.states[..., :2]
threshold = 0.25
values0 = jnp.linalg.norm(positions, axis=-1) - threshold # circle: less than 5 is in the target set 

# Choose solver, pick either backwards reachable set or backwards reachable tube

In [7]:
# for just backwards reachable set
# solver_settings = hj.SolverSettings.with_accuracy("very_high")

# for backwards reachable tube
solver_settings = hj.SolverSettings.with_accuracy("very_high", hamiltonian_postprocessor=hj.solver.backwards_reachable_tube)

# Solve for the value function

In [ ]:
all_values = hj.solve(solver_settings, dynamics, grid, times, values0)

  1%|4                                |  0.0139/1.0 [00:12<15:14, 927.67s/sim_s]

In [ ]:
value_function = np.array(all_values[-1]) # brs at -T

# Plot value function and the values less than zero (black)

In [ ]:
x3_target = np.pi / 2
x3_index = np.argmin(np.abs(grid.coordinate_vectors[2]-x3_target))
vf_slice = value_function[:, :, x3_index]

plt.figure(figsize=(7,6))
cf = plt.contourf(grid.coordinate_vectors[0], grid.coordinate_vectors[1], vf_slice.T, levels=60)
plt.contour(grid.coordinate_vectors[0], grid.coordinate_vectors[1], vf_slice.T, levels=[0], colors="black", linewidths=2)
plt.xlabel("x1")
plt.ylabel("x2")
plt.colorbar(cf, label=f"V(x1, x2, x3={np.round(x3_target, 2)}, {time})")
plt.title(f"BRT T={time}")
plt.show()